# Fantasy High-Performer Classifier Training
Trains an XGBoost binary classifier to predict the probability that a player would rank among the top 11 fantasy scorers *in that specific match*, from the same pre-match features used by the regression model. Reads `data/processed/fantasy_classifier_dataset.csv` (produced by `scripts/fantasy_classifier_dataset.py`, which reuses `scripts/dataset_builder.py`'s regression dataset and labels a player `1` if their actual fantasy points ranked in the match's top 11, else `0` — see that script's module docstring for the labeling rationale).

## 1. Load & inspect the dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
import joblib
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve, classification_report,
)
from google.colab import files

DATA_PATH = "fantasy_classifier_dataset.csv"
df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
print(df.shape)
df.head()

In [ ]:
df.describe(include="all").T


In [ ]:
TARGET_COL = "is_high_performer"

class_counts = df[TARGET_COL].value_counts().sort_index()
print("Class balance:")
print(class_counts)
print(f"Positive rate: {df[TARGET_COL].mean():.1%}")

class_counts.plot(kind="bar", figsize=(4, 4))
plt.title("is_high_performer class balance")
plt.xticks([0, 1], ["Not top-11 in match", "Top-11 in match"], rotation=0)
plt.show()

print("\nMissing values per column (top 15):")
print(df.isna().sum().sort_values(ascending=False).head(15))

## 2. Separate features / target, encode categoricals
Same exclusion logic as `train_fantasy_model.ipynb`, plus `fantasy_points` itself (and `batting_points` / `bowling_points` / `fielding_points`, its components) — those are the source the label was derived from in `fantasy_classifier_dataset.py` and must never leak into the classifier as an input.

In [ ]:
ID_COLS = ["match_id", "player", "date",
           "batting_points", "bowling_points", "fielding_points",
           "fantasy_points"]  # fantasy_points is the label's source, never a feature

CATEGORICAL_COLS = [c for c in ["venue", "batting_team", "bowling_team", "phase",
                                 "player_role", "player_sub_role"] if c in df.columns]

feature_cols = [c for c in df.columns if c not in ID_COLS + [TARGET_COL]]

encoders = {}
for col in CATEGORICAL_COLS:
    df[col] = df[col].fillna("Unknown").astype(str)
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

X = df[feature_cols]
y = df[TARGET_COL]
print(f"{len(feature_cols)} features, target = '{TARGET_COL}'")
feature_cols[:20]

## 3. Train / validation / test split — stratified by season AND label
Same per-season random-split rationale as `train_fantasy_model.ipynb` (proportional representation of every season in train/val/test, rather than only validating on the newest 1-2 seasons). Additionally stratified on `is_high_performer` within each season split — with the within-match top-11 label, class balance sits close to 11/(pool size per match) rather than a fixed percentile, so it isn't guaranteed to be perfectly even; stratifying still protects val/test from drifting to a meaningfully different positive rate than train, especially for smaller seasons.

In [ ]:
train_idx, val_idx, test_idx = [], [], []

for season in sorted(df["season"].unique()):
    season_df = df[df["season"] == season]
    season_idx = season_df.index
    season_y = season_df[TARGET_COL]

    # Stratify only if both classes are present with enough members; else
    # fall back to a plain random split for that season (a season with a
    # handful of matches / only one class present can't be stratified).
    can_stratify = season_y.nunique() > 1 and season_y.value_counts().min() >= 4

    train, temp = train_test_split(
        season_idx,
        test_size=0.30,
        random_state=42,
        shuffle=True,
        stratify=season_y if can_stratify else None,
    )
    temp_y = df.loc[temp, TARGET_COL]
    can_stratify_temp = temp_y.nunique() > 1 and temp_y.value_counts().min() >= 2
    val, test = train_test_split(
        temp,
        test_size=0.50,
        random_state=42,
        shuffle=True,
        stratify=temp_y if can_stratify_temp else None,
    )

    train_idx.extend(train)
    val_idx.extend(val)
    test_idx.extend(test)

X_train = X.loc[train_idx].reset_index(drop=True)
y_train = y.loc[train_idx].reset_index(drop=True)

X_val = X.loc[val_idx].reset_index(drop=True)
y_val = y.loc[val_idx].reset_index(drop=True)

X_test = X.loc[test_idx].reset_index(drop=True)
y_test = y.loc[test_idx].reset_index(drop=True)

print(f"Train      : {len(X_train):,}   positive rate={y_train.mean():.1%}")
print(f"Validation : {len(X_val):,}   positive rate={y_val.mean():.1%}")
print(f"Test       : {len(X_test):,}   positive rate={y_test.mean():.1%}")

## 4. Hyperparameter tuning (randomized search, val-set early stopping inside CV)

In [ ]:
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"scale_pos_weight = {scale_pos_weight:.3f}")

param_distributions = {
    "n_estimators": [300, 500, 800, 1200],
    "max_depth": [3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.02, 0.03, 0.05, 0.08],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5, 7, 10],
    "reg_lambda": [0.5, 1.0, 1.5, 2.0],
    "reg_alpha": [0.0, 0.05, 0.1, 0.3],
}

base_clf = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
)

search = RandomizedSearchCV(
    base_clf,
    param_distributions=param_distributions,
    n_iter=40,
    scoring="average_precision",  # PR-AUC — more informative than accuracy on an imbalanced label
    cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
search.fit(X_train, y_train)

print("Best CV average precision (PR-AUC):", search.best_score_)
print("Best params:")
for k, v in search.best_params_.items():
    print(f"  {k}: {v}")

## 5. Refit best params with early stopping on the validation set

In [ ]:
best_params = search.best_params_

model = xgb.XGBClassifier(
    **best_params,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
    early_stopping_rounds=50,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=50,
)
print(f"Best iteration: {model.best_iteration}")

## 6. Evaluate — Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC

In [ ]:
def evaluate(name, X_split, y_split):
    proba = model.predict_proba(X_split)[:, 1]
    preds = (proba >= 0.5).astype(int)

    acc = accuracy_score(y_split, preds)
    prec = precision_score(y_split, preds, zero_division=0)
    rec = recall_score(y_split, preds, zero_division=0)
    f1 = f1_score(y_split, preds, zero_division=0)
    roc_auc = roc_auc_score(y_split, proba)
    pr_auc = average_precision_score(y_split, proba)

    print(f"{name:>6}  Acc={acc:.3f}  Prec={prec:.3f}  Rec={rec:.3f}  "
          f"F1={f1:.3f}  ROC-AUC={roc_auc:.3f}  PR-AUC={pr_auc:.3f}")
    return proba, preds, dict(accuracy=acc, precision=prec, recall=rec,
                               f1=f1, roc_auc=roc_auc, pr_auc=pr_auc)

_ = evaluate("Train", X_train, y_train)
_ = evaluate("Val", X_val, y_val)
test_proba, test_preds, test_metrics = evaluate("Test", X_test, y_test)

print("\nTest classification report:")
print(classification_report(y_test, test_preds, target_names=["Not high performer", "High performer"]))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

cm = confusion_matrix(y_test, test_preds)
axes[0].imshow(cm, cmap="Blues")
axes[0].set_title("Test confusion matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(["Not high", "High"])
axes[0].set_yticks([0, 1]); axes[0].set_yticklabels(["Not high", "High"])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha="center", va="center")

fpr, tpr, _ = roc_curve(y_test, test_proba)
axes[1].plot(fpr, tpr, label=f"ROC-AUC={test_metrics['roc_auc']:.3f}")
axes[1].plot([0, 1], [0, 1], "r--")
axes[1].set_title("Test ROC curve")
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].legend()

prec_curve, rec_curve, _ = precision_recall_curve(y_test, test_proba)
axes[2].plot(rec_curve, prec_curve, label=f"PR-AUC={test_metrics['pr_auc']:.3f}")
axes[2].set_title("Test Precision-Recall curve")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")
axes[2].legend()

plt.tight_layout()
plt.show()

## 7. Feature importance & SHAP

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances.head(25).plot(kind="barh", figsize=(8, 8))
plt.gca().invert_yaxis()
plt.title("XGBoost feature importance (top 25)")
plt.show()
importances.head(25)

In [ ]:
explainer = shap.TreeExplainer(model)
sample = X_test.sample(min(1000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(sample)
shap.summary_plot(shap_values, sample, max_display=20)

## 8. Save model artifacts
Saves everything the prediction pipeline needs: the classifier itself, the label encoders (shared naming convention with the regression model's `encoders.pkl`, but saved separately here so the two models' encodings can't accidentally cross-contaminate if one pipeline changes categorical handling later), the exact feature column order, and evaluation metrics.

In [ ]:
model.save_model(ARTIFACT_DIR / "fantasy_classifier.json")
joblib.dump(model, ARTIFACT_DIR / "fantasy_classifier.pkl")
joblib.dump(encoders, ARTIFACT_DIR / "classifier_encoders.pkl")
joblib.dump(feature_cols, ARTIFACT_DIR / "classifier_feature_columns.pkl")

metrics = {
    "best_params": best_params,
    "best_iteration": int(model.best_iteration),
    "train": evaluate("Train", X_train, y_train)[2],
    "val": evaluate("Val", X_val, y_val)[2],
    "test": test_metrics,
    "n_features": len(feature_cols),
    "trained_rows": int(len(X_train)),
    "positive_rate_train": float(y_train.mean()),
}
joblib.dump(metrics, ARTIFACT_DIR / "classifier_metrics.pkl")
print("Saved model artifacts to", ARTIFACT_DIR)
metrics